# Using FastMCP with Claude: Ergonomics & Advanced Primitives

This notebook demonstrates **how to use FastMCP with Claude**. FastMCP is the high-level, ergonomic framework shipped natively in the Model Context Protocol SDK. It eliminates routing boilerplate and makes it incredibly easy to harness all three MCP primitives: tools, resources, and prompts.

By the end you will have:
- A FastMCP server exposing **tools**, **resources**, and **prompts**
- A client that reads resources, fetches prompt templates, and runs the agentic tool-use loop
- Working examples of **`Context`** for server-side logging and progress reporting

> **Note**: This notebook covers advanced concepts. If you're completely new to the Model Context Protocol, consider reading the official MCP introduction first.

> **Note on outputs**: The cell outputs below are illustrative mock outputs, embedded so you can read the complete flow without running the server locally. If any output looks unexpected, re-run the notebook with a live API key for ground-truth results.

---

## Prerequisites

| Requirement | Details |
|---|---|
| Python | 3.10 or later |
| Anthropic API key | [Get one here](https://console.anthropic.com/settings/keys) |

Store your key in a `.env` file next to this notebook:

```
ANTHROPIC_API_KEY=your-key-here
```

## 1. Setup

In [1]:
%%capture
%pip install mcp anthropic python-dotenv

## 2. The FastMCP Server

This server demonstrates **all three MCP primitives** that FastMCP supports:

| Primitive | Decorator | Purpose |
|---|---|---|
| **Tool** | `@mcp.tool()` | Let the LLM *do* things (side effects, computation) |
| **Resource** | `@mcp.resource()` | Let the LLM *read* data (like GET endpoints) |
| **Prompt** | `@mcp.prompt()` | Provide reusable prompt templates |

We also inject a **`Context`** object into one tool to show server-side logging.

In [2]:
%%writefile knowledge_server.py
"""
knowledge_server.py — FastMCP server with tools, resources, and prompts.
"""
from mcp.server.fastmcp import Context, FastMCP

mcp = FastMCP("KnowledgeServer")

# ---------------------------------------------------------------------------
# Static dataset
# ---------------------------------------------------------------------------
_DOCS: dict[str, str] = {
    "python": (
        "Python is a high-level, interpreted programming language "
        "known for its readability and versatility. Created by "
        "Guido van Rossum, first released in 1991."
    ),
    "rust": (
        "Rust is a systems programming language focused on safety, "
        "speed, and concurrency. It achieves memory safety without "
        "garbage collection."
    ),
    "mcp": (
        "The Model Context Protocol (MCP) is an open standard that "
        "lets AI applications connect to external data sources and "
        "tools in a unified way."
    ),
}

# ---------------------------------------------------------------------------
# Resources — read-only data the LLM can load into its context
# ---------------------------------------------------------------------------


@mcp.resource("docs://topics")
def list_topics() -> str:
    """List all available documentation topics."""
    return ", ".join(_DOCS.keys())


@mcp.resource("docs://topic/{name}")
def get_topic(name: str) -> str:
    """Get documentation for a specific topic."""
    return _DOCS.get(
        name.lower(), f"No docs for '{name}'."
    )


# ---------------------------------------------------------------------------
# Tools — actions the LLM can execute
# ---------------------------------------------------------------------------


@mcp.tool()
def search_docs(query: str) -> str:
    """Search documentation by keyword. Returns matching topics."""
    q = query.lower()
    matches = [
        f"{name}: {text}"
        for name, text in _DOCS.items()
        if q in text.lower() or q in name
    ]
    if not matches:
        return f"No results for '{query}'."
    return "\n".join(matches)


@mcp.tool()
async def summarize_topic(
    name: str, ctx: Context
) -> str:
    """Summarize a topic with server-side logging via Context."""
    await ctx.info(f"Summarizing topic: {name}")
    text = _DOCS.get(name.lower())
    if text is None:
        await ctx.warning(f"Topic not found: {name}")
        return f"Topic '{name}' not found."
    await ctx.info("Summary complete")
    words = text.split()
    short = " ".join(words[:12])
    return f"{name.title()} (summary): {short}..."


# ---------------------------------------------------------------------------
# Prompts — reusable templates for common interactions
# ---------------------------------------------------------------------------


@mcp.prompt(name="topic_expert", description="Expert on a given topic")
def topic_expert(topic: str) -> str:
    """Generate a system prompt for a topic expert."""
    return (
        f"You are an expert on {topic}. Answer questions "
        f"about {topic} clearly and concisely, using the "
        f"documentation available through your tools."
    )


@mcp.prompt(name="compare_topics", description="Compare two topics")
def compare_topics(
    topic_a: str, topic_b: str
) -> str:
    """Generate a prompt to compare two topics."""
    return (
        f"Compare and contrast {topic_a} and {topic_b}. "
        f"Use the available documentation tools to look up "
        f"each topic before answering."
    )


if __name__ == "__main__":
    mcp.run()

Writing knowledge_server.py


## 3. The MCP Client

This client extends the pattern from the native MCP notebook with two new capabilities:

- **`list_resources()` / `read_resource()`** — browse and load server data into the conversation
- **`list_prompts()` / `get_prompt()`** — fetch reusable prompt templates from the server

The agentic tool-use loop remains exactly the same.

In [3]:
import asyncio
import os
from contextlib import AsyncExitStack
from typing import Optional

import anthropic
from dotenv import load_dotenv
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

load_dotenv()

# https://docs.anthropic.com/en/docs/about-claude/models/overview
MODEL = "claude-haiku-4-5"
SERVER_SCRIPT = "knowledge_server.py"


class MCPClient:
    """MCP client with resource and prompt support."""

    def __init__(self) -> None:
        self.session: Optional[ClientSession] = None
        self.exit_stack = AsyncExitStack()
        self.anthropic = anthropic.Anthropic(
            api_key=os.environ.get("ANTHROPIC_API_KEY")
        )

    async def connect(self, server_script: str) -> None:
        """Launch the server and open an MCP session."""
        params = StdioServerParameters(
            command="python",
            args=[server_script],
        )
        read, write = await self.exit_stack.enter_async_context(
            stdio_client(params)
        )
        self.session = await self.exit_stack.enter_async_context(
            ClientSession(read, write)
        )
        await self.session.initialize()

        # Discover capabilities
        tools = (await self.session.list_tools()).tools
        resources = (await self.session.list_resources()).resources
        prompts_list = (await self.session.list_prompts()).prompts

        print(f"Connected to '{server_script}'")
        print(f"  Tools:     {[t.name for t in tools]}")
        print(f"  Resources: {[r.uri for r in resources]}")
        print(f"  Prompts:   {[p.name for p in prompts_list]}")

    # ---- Resource helpers ----

    async def read_resource(self, uri: str) -> str:
        """Read a resource by URI and return its text."""
        result = await self.session.read_resource(uri)
        return result.contents[0].text

    # ---- Prompt helpers ----

    async def get_prompt(
        self, name: str, **kwargs: str
    ) -> str:
        """Fetch a prompt template and return its text."""
        result = await self.session.get_prompt(
            name, arguments=kwargs
        )
        return result.messages[0].content.text

    # ---- Tool-use loop (same as native notebook) ----

    async def _get_tools(self) -> list[dict]:
        response = await self.session.list_tools()
        return [
            {
                "name": t.name,
                "description": t.description,
                "input_schema": t.inputSchema,
            }
            for t in response.tools
        ]

    async def query(
        self,
        user_message: str,
        system: str = "",
    ) -> str:
        """Send a query with optional system prompt."""
        tools = await self._get_tools()
        messages: list[dict] = [
            {"role": "user", "content": user_message}
        ]
        kwargs: dict = {
            "model": MODEL,
            "max_tokens": 1024,
            "tools": tools,
            "messages": messages,
        }
        if system:
            kwargs["system"] = system

        while True:
            response = self.anthropic.messages.create(
                **kwargs
            )
            # After first call, remove system to avoid resending
            kwargs.pop("system", None)

            tool_calls = [
                b for b in response.content
                if b.type == "tool_use"
            ]
            if not tool_calls:
                return "\n".join(
                    b.text for b in response.content
                    if b.type == "text"
                )

            messages.append(
                {"role": "assistant", "content": response.content}
            )

            tool_results = []
            for call in tool_calls:
                print(
                    f"  -> tool '{call.name}'"
                    f" args={call.input}"
                )
                result = await self.session.call_tool(
                    call.name, call.input
                )
                tool_results.append(
                    {
                        "type": "tool_result",
                        "tool_use_id": call.id,
                        "content": result.content,
                    }
                )
            messages.append(
                {"role": "user", "content": tool_results}
            )

    async def close(self) -> None:
        await self.exit_stack.aclose()

## 4. Demos

### Demo A — Reading resources
We read the `docs://topics` resource to see what the server knows, then read a specific topic.

### Demo B — Using a prompt template
We fetch the `topic_expert` prompt from the server and pass it as the `system` parameter so Claude adopts that persona.

### Demo C — Tool use with Context logging
The `summarize_topic` tool uses `Context` on the server side to emit log messages as it works.

In [4]:
async def demo() -> None:
    client = MCPClient()
    try:
        await client.connect(SERVER_SCRIPT)

        # -- Demo A: Resources --
        print("\n=== Demo A: Reading resources ===")
        topics = await client.read_resource("docs://topics")
        print(f"Available topics: {topics}")

        mcp_doc = await client.read_resource(
            "docs://topic/mcp"
        )
        print(f"MCP doc: {mcp_doc}")

        # -- Demo B: Prompt template --
        print("\n=== Demo B: Prompt template as system message ===")
        sys_prompt = await client.get_prompt(
            "topic_expert", topic="Rust"
        )
        print(f"System prompt: {sys_prompt}")
        answer = await client.query(
            "What makes Rust special?",
            system=sys_prompt,
        )
        print(f"\nClaude: {answer}")

        # -- Demo C: Context logging --
        print("\n=== Demo C: Tool with Context logging ===")
        answer = await client.query(
            "Give me a quick summary of Python."
        )
        print(f"\nClaude: {answer}")

    finally:
        await client.close()


asyncio.run(demo())

Connected to 'knowledge_server.py'
  Tools:     ['search_docs', 'summarize_topic']
  Resources: ['docs://topics']
  Prompts:   ['topic_expert', 'compare_topics']

=== Demo A: Reading resources ===
Available topics: python, rust, mcp
MCP doc: The Model Context Protocol (MCP) is an open standard that lets AI applications connect to external data sources and tools in a unified way.

=== Demo B: Prompt template as system message ===
System prompt: You are an expert on Rust. Answer questions about Rust clearly and concisely, using the documentation available through your tools.
  -> tool 'search_docs' args={'query': 'rust'}

Claude: Rust is a systems programming language focused on safety, speed, and concurrency. Its key innovation is achieving memory safety without garbage collection, using a unique ownership system checked at compile time.

=== Demo C: Tool with Context logging ===
  -> tool 'summarize_topic' args={'name': 'python'}

Claude: Here's a quick summary of Python: it's a high-l

## 5. How It Works

### The three primitives side by side

```
FastMCP Server                          Client
+---------------------------------+     +------------------------+
|                                 |     |                        |
|  @mcp.resource("docs://topics") | <-- | session.read_resource()|
|  @mcp.resource("docs://topic/{name}")  |                        |
|                                 |     |                        |
|  @mcp.tool()                    | <-- | session.call_tool()    |
|    search_docs(query)           |     |                        |
|    summarize_topic(name, ctx)   |     |                        |
|                                 |     |                        |
|  @mcp.prompt()                  | <-- | session.get_prompt()   |
|    topic_expert(topic)          |     |                        |
|    compare_topics(a, b)         |     |                        |
+---------------------------------+     +------------------------+
```

### When to use each primitive

| Primitive | Analogy | Use when... |
|---|---|---|
| **Resource** | GET endpoint | You want the LLM to *read* data without side effects |
| **Tool** | POST endpoint | You want the LLM to *do* something (compute, write, call APIs) |
| **Prompt** | Template | You want to share reusable interaction patterns with clients |

### Context injection

Adding `ctx: Context` to any tool or resource function gives you:

- **Logging**: `await ctx.info(...)`, `ctx.warning(...)`, `ctx.debug(...)`
- **Progress**: `await ctx.report_progress(0.5, total=1.0)`
- **Resource access**: `await ctx.read_resource(uri)` to read other resources from within a tool

The `Context` parameter is automatically injected by FastMCP — it never comes from the LLM's tool arguments.

## Next Steps

- **Structured output** — return Pydantic models or TypedDicts from tools for validated, typed responses
- **Lifespan management** — use `@asynccontextmanager` to initialize databases or HTTP clients on server startup
- **Remote transport** — swap `stdio` for `streamable-http` to run the server on a different machine
- **Elicitation** — use `await ctx.elicit(...)` to request additional information from the user mid-tool

### References

- [MCP Python SDK](https://github.com/modelcontextprotocol/python-sdk)
- [FastMCP server docs](https://github.com/modelcontextprotocol/python-sdk#server)
- [MCP specification — Resources](https://modelcontextprotocol.io/docs/concepts/resources)
- [MCP specification — Prompts](https://modelcontextprotocol.io/docs/concepts/prompts)
- [Anthropic tool use docs](https://docs.anthropic.com/en/docs/agents-and-tools/tool-use/overview)